# Notebook 03 — Model Training

Trains all four classifiers: Logistic Regression, Decision Tree, Random Forest, SVM.
Models are saved to `models/` as `.joblib` files.


In [ ]:
import sys
sys.path.insert(0, '..')

from src.malaria_forecast.config import load_config
from src.malaria_forecast.data_loader import load_raw_dataset
from src.malaria_forecast.preprocessing import preprocess_data
from src.malaria_forecast.models import build_models
from src.malaria_forecast.artifacts import save_artifact
from pathlib import Path

config = load_config('../config/config.yaml')
df = load_raw_dataset('../dataset/Malaria_Dataset.csv')
result = preprocess_data(df, config)

X_train, X_test = result['X_train'], result['X_test']
y_train, y_test = result['y_train'], result['y_test']
print('Data ready. Training...')

## Train All Four Models

In [ ]:
classifiers = build_models(config)

fitted = {}
for name, clf in classifiers.items():
    print(f'  Fitting {name}...')
    clf.fit(X_train, y_train)
    fitted[name] = clf
    print(f'  ✅ {name} done')

print('\nAll models trained.')

## Save Models to Disk

In [ ]:
models_dir = Path('../models')
models_dir.mkdir(exist_ok=True)

for name, clf in fitted.items():
    path = models_dir / f'{name}.joblib'
    save_artifact(clf, path)
    print(f'  Saved: {path}')

## Save Pipeline Metadata

In [ ]:
from src.malaria_forecast.features import LABEL_MAP

metadata = {
    'scaler': result['scaler'],
    'encoder': result['encoder'],
    'imputation_defaults': result['imputation_defaults'],
    'final_feature_order': result['final_feature_order'],
    'best_model_name': 'svm',
    'best_model_file': 'svm.joblib',
    'label_map': LABEL_MAP,
}
save_artifact(metadata, models_dir / 'metadata.joblib')
print('Pipeline metadata saved.')

## Quick Accuracy Sanity Check

In [ ]:
from sklearn.metrics import accuracy_score

print(f'{'Model':<25} {'Train Acc':>10} {'Test Acc':>10}')
print('-' * 47)
for name, clf in fitted.items():
    tr = accuracy_score(y_train, clf.predict(X_train))
    te = accuracy_score(y_test,  clf.predict(X_test))
    print(f'{name:<25} {tr:>9.4f}  {te:>9.4f}')